# SVM con kernels (solucion)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_wine


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_NOTEBOOKS = "https://github.com/AdanReyes2407/Notebooks_NM_ML.git"


def is_colab():
    return "google.colab" in sys.modules


def ensure_repo_clone(repo_url=REPO_NOTEBOOKS, target=Path('/content/Notebooks_NM_ML')):
    if is_colab() and not target.exists():
        print(f"Clonando repositorio en {target} ...")
        subprocess.run(["git", "clone", repo_url, str(target)], check=True)


def resolve_data_dir():
    ensure_repo_clone()
    candidates = [
        Path('/content/Notebooks_NM_ML/Bases de datos'),
        Path.cwd() / 'Bases de datos',
        Path.cwd().parent / 'Bases de datos',
        Path('/content/Bases de datos'),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


In [ ]:
def load_dataset(data_dir):
    if data_dir is not None:
        p = data_dir / 'column_2C.dat'
        if p.exists():
            df = pd.read_csv(p, delim_whitespace=True, header=None)
            X = df.iloc[:, :6].copy()
            y_raw = df.iloc[:, 6].astype(str).str.upper().str.strip()
            y = (y_raw == 'ABNORMAL').astype(int)
            return X, y, 'column_2C.dat'

    wine = load_wine(as_frame=True)
    X = wine.data.copy()
    y = wine.target.astype(int).copy()
    return X, y, 'sklearn_wine'


DATA_DIR = resolve_data_dir()
X, y, fuente = load_dataset(DATA_DIR)
print('Fuente:', fuente)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=16, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'lineal': SVC(kernel='linear', C=1.0),
    'RBF': SVC(kernel='rbf', C=3.0, gamma='scale'),
    'polinomial': SVC(kernel='poly', degree=3, C=1.0, gamma='scale'),
}

rows = []
for name, m in models.items():
    m.fit(X_train_s, y_train)
    y_hat = m.predict(X_test_s)
    rows.append((name, accuracy_score(y_test, y_hat)))

res = pd.DataFrame(rows, columns=['kernel', 'accuracy']).sort_values('accuracy', ascending=False)
print(res)


In [ ]:
rows = []
best = (None, -1)
for C in [0.3, 1, 3, 10]:
    for gamma in ['scale', 0.3, 0.1, 0.03]:
        m = SVC(kernel='rbf', C=C, gamma=gamma)
        m.fit(X_train_s, y_train)
        acc = accuracy_score(y_test, m.predict(X_test_s))
        rows.append((C, gamma, acc))
        if acc > best[1]:
            best = ((C, gamma), acc)

sweep = pd.DataFrame(rows, columns=['C', 'gamma', 'accuracy']).sort_values('accuracy', ascending=False)
print(sweep.head(10))
print('Mejor configuracion RBF:', best)
